**CHECK SQL AGAINST PANDAS**

In [1]:
import sqlite3
import numpy as np
import pandas as pd

df = pd.read_csv("train.csv")
conn = sqlite3.connect("housing.db")
df.to_sql("houses", conn, if_exists = "replace", index = False)

def q(sql):
  return pd.read_sql(sql, conn)

**AVERAGE PRICE BY NEIGHBORHOOD**

In [2]:
sql_result = q("SELECT Neighborhood, AVG(SalePrice) AS avg_price FROM houses GROUP BY Neighborhood ORDER BY Neighborhood")
pandas_result = df.groupby("Neighborhood")["SalePrice"].mean().reset_index().sort_values("Neighborhood")

print(sql_result["Neighborhood"].tolist() == pandas_result["Neighborhood"].tolist())
print(np.allclose(sql_result["avg_price"].values, pandas_result["SalePrice"].values))

True
True


**HOUSES OVER 400,000 WITH 4 OR MORE BEDROOMS**

In [3]:
sql_n = q("SELECT COUNT(*) AS n FROM houses WHERE SalePrice > 400000 AND BedroomAbvGr >=4")["n"][0]
pandas_n = len(df[(df["SalePrice"] > 400000) & (df["BedroomAbvGr"] >=4)])
print(sql_n, pandas_n, sql_n == pandas_n)

14 14 True


**AVERAGE PRICE AT QUALITY 10**

In [4]:
sql_avg = q("SELECT AVG(SalePrice) AS a FROM houses WHERE OverallQual = 10")["a"][0]
pandas_avg = df[df["OverallQual"] == 10]["SalePrice"].mean()
print(round(sql_avg, 2), round(pandas_avg, 2))

438588.39 438588.39


**FIVE PRACTICE PROBLEMS**

In [5]:
# Big Neighborhoods
q("""
SELECT Neighborhood, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY Neighborhood
HAVING COUNT(*) >= 100
ORDER BY avg_price DESC
""")

,Neighborhood,houses,avg_price
0,CollgCr,150,197966.0
1,NAmes,225,145847.0
2,OldTown,113,128225.0
3,Edwards,100,128220.0


*Four neighborhoods have at least 100 houses. CollgCr has the highest average price (197,966), and OldTown and Edwards are nearly identical (about 128,200). NAmes is the largest, with 225 houses, but a mid-range price.*

In [6]:
# Find the top 5 neighborhoods by average price per square foot of living area
q("""
SELECT Neighborhood, ROUND(AVG(SalePrice * 1.0 / GrLivArea), 1) AS price_per_sqft
FROM houses
GROUP BY Neighborhood
ORDER BY price_per_sqft DESC
LIMIT 5
""")

,Neighborhood,price_per_sqft
0,StoneBr,165.4
1,NridgHt,164.6
2,Veenker,155.0
3,Somerst,141.3
4,Timber,139.8


*StoneBr and NridgHt have the highest average price per square foot (about 165), well above the typical figure of about 120 from my earlier work, so the same space costs about a third more there. NoRidge has the highest average price but is not in the top five per square foot, because its houses are large.*

In [7]:
#Show the number of houses and the average price for each value of GarageCars sorted by GarageCars
q("""
SELECT GarageCars, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY GarageCars
ORDER BY GarageCars
""" )

,GarageCars,houses,avg_price
0,0,81,103317.0
1,1,369,128117.0
2,2,824,183852.0
3,3,181,309636.0
4,4,5,192656.0


*Average price rises with garage size from 0 to 3 cars (103,317 to 309,636, about 3 times). The 4-car group has only 5 houses, so its average is unreliable. Garage size also goes with house size and quality, so it doesn’t explain price on its own.*

In [8]:
# Central Air
q("""
SELECT CentralAir, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY CentralAir
ORDER BY CentralAir
""")

,CentralAir,houses,avg_price
0,N,95,105264.0
1,Y,1365,186187.0


*1,365 of 1,460 houses (about 93.5%) have central air, and they average 186,187 against 105,264 for the 95 houses without it, about 77% higher. Central air likely reflects age and quality as well as the feature itself.*

In [9]:
# Using JOIN
conn.execute("DROP TABLE IF EXISTS hood_stats")
conn.execute("""
CREATE TABLE hood_stats AS
SELECT Neighborhood, COUNT(*) AS houses, ROUND(AVG(SalePrice)) AS avg_price
FROM houses
GROUP BY Neighborhood
""")
conn.commit()

In [10]:
q("""
SELECT h.Neighborhood, COUNT(*) AS strong_houses
FROM houses AS h
JOIN hood_stats AS s ON h.Neighborhood = s.Neighborhood
WHERE h.OverallQual >= 8 AND h.SalePrice > s.avg_price
GROUP BY h.Neighborhood
ORDER BY strong_houses DESC
LIMIT 5
""" )

,Neighborhood,strong_houses
0,NridgHt,38
1,Somerst,28
2,CollgCr,25
3,Timber,14
4,NoRidge,13


*The neighborhoods with the most high-quality houses (quality 8 or higher) that sold above their own neighborhood average are NridgHt (38), Somerst (28), and CollgCr (25). NoRidge has the highest average price but only 13, because it has few houses and a high bar to beat. The counts favor larger neighborhoods, so a share would be a fairer comparison.*